### My implementation of penalized regressions
In this notebook, I provide improvements and extensions to the original implementation of penalized regressions presented in the book. These include:
* Bayesian optimization of hyperparameters using optuna.
* Walk-forward optimization.
* Additional Feature Engineering: including lags.

  
For training time purposes, I reduce the number of features (use features_short) and use only 20% of the stock_ids_short list. I also perform training using ElasticNet only.

I do not provide code on sparse portfolio optimization.

In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
np.random.seed(123)
os.chdir('C:\\Users\\ferna\\OneDrive\\Downloads\\Code\\Python\\ML_Factor')

### 1) Data

In [90]:
from data.data_load import *

In [91]:
# Selection of 20% of stock_ids_short (Careful in real application, has survivorship bias. I use only for simplification).
stock_ids_selected = selected = random.sample(stock_ids_short, k=int(0.2 * len(stock_ids_short)))
len(stock_ids_selected)

158

In [92]:
# Filtering for stock_ids_selected and features_short
data_ml_selected = data_ml.loc[data_ml['stock_id'].isin(stock_ids_selected), ['date', 'stock_id','R1M_Usd'] + features_short]
data_ml_selected = data_ml_selected.sort_values(by=['date', 'stock_id'], ascending=True)
data_ml_selected.index = range(data_ml_selected.shape[0])
data_ml_selected.iloc[[0,1,2,-2,-1]]

,date,stock_id,R1M_Usd,Div_Yld,Eps,Mkt_Cap_12M_Usd,Mom_11M_Usd,Ocf,Pb,Vol1Y_Usd
0,2000-01-31,24,0.222,0.12,0.32,0.29,0.50,0.20,0.66,0.68
1,2000-01-31,34,0.587,0.16,0.15,0.17,0.47,0.22,0.64,0.88
2,2000-01-31,41,0.051,0.73,0.78,0.54,0.42,0.54,0.61,0.15
36022,2018-12-31,1195,0.157,0.24,0.64,0.77,0.50,0.76,0.62,0.49
36023,2018-12-31,1212,0.222,0.13,0.99,0.72,0.09,0.87,0.73,0.88


In [93]:
# Adding lags of dependent variable - may introduce endogeneity if the errors are serially correlated.
# Add 4 lags
data_ml_selected_lags = data_ml_selected.copy()
nlags = 4
for ll in range(1,nlags+1):
    data_ml_selected_lags[f'Lag{ll}_R1M_Usd'] = data_ml_selected.groupby('stock_id')['R1M_Usd'].shift(ll)

In [94]:
# REMOVE NAN
data_ml_selected_lags = data_ml_selected_lags.dropna()

In [95]:
data_ml_selected_lags.iloc[[0,1,2,-2,-1]]

,date,stock_id,R1M_Usd,Div_Yld,Eps,Mkt_Cap_12M_Usd,Mom_11M_Usd,Ocf,Pb,Vol1Y_Usd,Lag1_R1M_Usd,Lag2_R1M_Usd,Lag3_R1M_Usd,Lag4_R1M_Usd
632,2000-05-31,24,0.083,0.21,0.36,0.31,0.87,0.36,0.65,0.76,-0.070,0.098,-0.026,0.222
633,2000-05-31,34,0.299,0.28,0.16,0.25,0.97,0.20,0.79,0.78,0.290,0.040,-0.048,0.587
634,2000-05-31,41,0.021,0.79,0.78,0.52,0.30,0.58,0.53,0.15,0.058,-0.178,0.183,0.051
36022,2018-12-31,1195,0.157,0.24,0.64,0.77,0.50,0.76,0.62,0.49,-0.180,0.050,-0.250,0.031
36023,2018-12-31,1212,0.222,0.13,0.99,0.72,0.09,0.87,0.73,0.88,-0.125,0.013,-0.266,0.044


### 2) Elastic Net training

In [96]:
import optuna
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import ElasticNet

In [97]:
# Functions for walk-forward optimization.

def prepare_data(data_ml_selected_lags, features, y_name , train_tuple, val_tuple, test_tuple):

    """
    Split panel data into training, validation, and test sets by date.
    Returns (X, y) pairs as arrays for each period.
    """
    
    idx_train = data_ml_selected_lags['date'].index[
        (data_ml_selected_lags['date'] >= train_tuple[0]) & 
        (data_ml_selected_lags['date'] < train_tuple[1])
    ]

    idx_val  = data_ml_selected_lags['date'].index[
        (data_ml_selected_lags['date'] >= val_tuple[0]) & 
        (data_ml_selected_lags['date'] < val_tuple[1])
    ]

    idx_test  = data_ml_selected_lags['date'].index[
        (data_ml_selected_lags['date'] >= test_tuple[0]) & 
        (data_ml_selected_lags['date'] <= test_tuple[1])
    ]

    X_train = data_ml_selected_lags.loc[idx_train,:][features].values
    X_val   = data_ml_selected_lags.loc[idx_val,:][features].values
    X_test  = data_ml_selected_lags.loc[idx_test,:][features].values

    y_train = data_ml_selected_lags.loc[idx_train,:][y_name].values
    y_val   = data_ml_selected_lags.loc[idx_val,:][y_name].values
    y_test  = data_ml_selected_lags.loc[idx_test,:][y_name].values

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def optimize_hyperparameters(X_train, y_train, X_val, y_val, n_trials=150):

    def objective(trial):

        alpha = trial.suggest_float("alpha", 1e-7, 1e2, log=True)
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

        model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=10000
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        return mean_squared_error(y_val, y_pred)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    return study
    

In [98]:
dates = data_ml_selected_lags['date'].sort_values().unique()

init_train = 120
init_val = 150
retrain_step = 6

features = data_ml_selected_lags.columns.drop(['date', 'stock_id', 'R1M_Usd'])
y_name = 'R1M_Usd'

results = []
hyperparameters = []

for i in range(0, len(dates) - init_val, retrain_step):

    print(i)

    # Preparando dados
    
    train_tuple = (dates[i], dates[i + init_train])
    val_tuple = (dates[i + init_train], dates[i + init_val])

    test_end = min(i + init_val + retrain_step, len(dates))
    test_tuple = (dates[i + init_val], dates[test_end - 1])

    train, val, test = prepare_data(
        data_ml_selected_lags, features, y_name,
        train_tuple, val_tuple, test_tuple
    )

    
    # Hyperparameter optimization

    study = optimize_hyperparameters(train[0], train[1], val[0], val[1], n_trials=200)
    
    best_alpha    = study.best_params["alpha"]
    best_l1_ratio = study.best_params["l1_ratio"]

    # Model: fit on train+val, test on test.
    
    X_fit = np.concatenate([train[0], val[0]], axis=0)
    y_fit = np.concatenate([train[1], val[1]], axis=0)

    model = ElasticNet(**study.best_params, max_iter=10000)
    model.fit(X_fit, y_fit)

    # OOS
    y_pred = model.predict(test[0])
    
    test_dates = dates[i + init_val : i + init_val + retrain_step]

    test_df = data_ml_selected_lags.loc[
        data_ml_selected_lags['date'].isin(test_dates),
        ['date', 'stock_id', y_name]
    ].copy()

    test_df['y_pred'] = y_pred
    results.append(test_df)

    # Store hyperparameters
    hyperparameters.append({
        'date': test_dates[0],
        **study.best_params,
        'validation_mse': study.best_value
    })

# Combine results
results_df = pd.concat(results, ignore_index=True)
hyperparameters_df = pd.DataFrame(hyperparameters)
    

[I 2026-10-08 19:49:11,084] A new study created in memory with name: no-name-7684a27d-b4ca-4205-b208-17d7d5d4d788
[I 2026-10-08 19:49:11,092] Trial 0 finished with value: 0.011727920757891362 and parameters: {'alpha': 1.719901313179744, 'l1_ratio': 0.407379015415652}. Best is trial 0 with value: 0.011727920757891362.
[I 2026-10-08 19:49:11,097] Trial 1 finished with value: 0.011727920757891362 and parameters: {'alpha': 0.1173388171856373, 'l1_ratio': 0.6278021787412205}. Best is trial 0 with value: 0.011727920757891362.
[I 2026-10-08 19:49:11,104] Trial 2 finished with value: 0.011978801899755536 and parameters: {'alpha': 0.007143080084788142, 'l1_ratio': 0.07983347377042715}. Best is trial 0 with value: 0.011727920757891362.
[I 2026-10-08 19:49:11,112] Trial 3 finished with value: 0.011727920757891362 and parameters: {'alpha': 1.0302114872942971, 'l1_ratio': 0.5900075844886727}. Best is trial 0 with value: 0.011727920757891362.
[I 2026-10-08 19:49:11,122] Trial 4 finished with value: 

0


[I 2026-10-08 19:49:11,282] Trial 20 finished with value: 0.012168235396680124 and parameters: {'alpha': 0.00017944784015637346, 'l1_ratio': 0.15702071909582255}. Best is trial 0 with value: 0.011727920757891362.
[I 2026-10-08 19:49:11,293] Trial 21 finished with value: 0.011727920757891362 and parameters: {'alpha': 1.1019284486855672, 'l1_ratio': 0.6003787383155171}. Best is trial 0 with value: 0.011727920757891362.
[I 2026-10-08 19:49:11,304] Trial 22 finished with value: 0.011727920757891362 and parameters: {'alpha': 13.343304413048958, 'l1_ratio': 0.7530503503574287}. Best is trial 0 with value: 0.011727920757891362.
[I 2026-10-08 19:49:11,313] Trial 23 finished with value: 0.011727920757891362 and parameters: {'alpha': 0.03636822476552237, 'l1_ratio': 0.47226452261269136}. Best is trial 0 with value: 0.011727920757891362.
[I 2026-10-08 19:49:11,321] Trial 24 finished with value: 0.011727920757891362 and parameters: {'alpha': 0.3118035276980204, 'l1_ratio': 0.5546281162355016}. Bes

6


[I 2026-10-08 19:49:13,408] Trial 20 finished with value: 0.010779957100354688 and parameters: {'alpha': 0.0007005535462404282, 'l1_ratio': 0.46534038497585184}. Best is trial 0 with value: 0.010587203443717287.
[I 2026-10-08 19:49:13,423] Trial 21 finished with value: 0.010587203443717287 and parameters: {'alpha': 0.3467624527471806, 'l1_ratio': 0.6145700605912499}. Best is trial 0 with value: 0.010587203443717287.
[I 2026-10-08 19:49:13,436] Trial 22 finished with value: 0.010587203443717287 and parameters: {'alpha': 1.1439655920530598, 'l1_ratio': 0.5181079990010701}. Best is trial 0 with value: 0.010587203443717287.
[I 2026-10-08 19:49:13,445] Trial 23 finished with value: 0.010587203443717287 and parameters: {'alpha': 19.22740278413437, 'l1_ratio': 0.6529649860156896}. Best is trial 0 with value: 0.010587203443717287.
[I 2026-10-08 19:49:13,454] Trial 24 finished with value: 0.010587203443717287 and parameters: {'alpha': 0.011084218350176244, 'l1_ratio': 0.9056478792288737}. Best 

...

[I 2026-10-08 19:49:30,528] Trial 22 finished with value: 0.01660768777695715 and parameters: {'alpha': 0.010435688853062924, 'l1_ratio': 0.49519292471619397}. Best is trial 19 with value: 0.016591968821566244.
[I 2026-10-08 19:49:30,538] Trial 23 finished with value: 0.01659180309990907 and parameters: {'alpha': 0.0006984930624187076, 'l1_ratio': 0.6380703050832495}. Best is trial 23 with value: 0.01659180309990907.
[I 2026-10-08 19:49:30,548] Trial 24 finished with value: 0.016605992063702373 and parameters: {'alpha': 4.924047737219532e-05, 'l1_ratio': 0.6661807121552853}. Best is trial 23 with value: 0.01659180309990907.
[I 2026-10-08 19:49:30,557] Trial 25 finished with value: 0.01660768777695715 and parameters: {'alpha': 0.1665212538309641, 'l1_ratio': 0.8303828306156894}. Best is trial 23 with value: 0.01659180309990907.
[I 2026-10-08 19:49:30,566] Trial 26 finished with value: 0.01660768777695715 and parameters: {'alpha': 0.06067856182093552, 'l1_ratio': 0.5408871683225822}. Bes

66


[I 2026-10-08 19:49:32,521] Trial 24 finished with value: 0.014752380079226224 and parameters: {'alpha': 8.247398463503304e-06, 'l1_ratio': 0.39727769785687805}. Best is trial 4 with value: 0.014685929817356962.
[I 2026-10-08 19:49:32,531] Trial 25 finished with value: 0.014736944578105024 and parameters: {'alpha': 6.606422346285988e-05, 'l1_ratio': 0.9033544412395229}. Best is trial 4 with value: 0.014685929817356962.
[I 2026-10-08 19:49:32,540] Trial 26 finished with value: 0.014686860618238791 and parameters: {'alpha': 0.0013779307715767733, 'l1_ratio': 0.549466242994655}. Best is trial 4 with value: 0.014685929817356962.
[I 2026-10-08 19:49:32,547] Trial 27 finished with value: 0.014704800919816203 and parameters: {'alpha': 0.05347069584517518, 'l1_ratio': 0.7444358948731767}. Best is trial 4 with value: 0.014685929817356962.
[I 2026-10-08 19:49:32,555] Trial 28 finished with value: 0.01469187559081659 and parameters: {'alpha': 0.0037884702466808814, 'l1_ratio': 0.316547761997657}.

72


[I 2026-10-08 19:49:34,434] Trial 23 finished with value: 0.20549910435388688 and parameters: {'alpha': 0.002188630512404353, 'l1_ratio': 0.1043427144104692}. Best is trial 11 with value: 0.2054330562824627.
[I 2026-10-08 19:49:34,443] Trial 24 finished with value: 0.20554306223842322 and parameters: {'alpha': 1.2569072099854885, 'l1_ratio': 0.29695438599029167}. Best is trial 11 with value: 0.2054330562824627.
[I 2026-10-08 19:49:34,452] Trial 25 finished with value: 0.20551177378306987 and parameters: {'alpha': 0.009643578180876607, 'l1_ratio': 0.18440099553721584}. Best is trial 11 with value: 0.2054330562824627.
[I 2026-10-08 19:49:34,459] Trial 26 finished with value: 0.20554306223842322 and parameters: {'alpha': 0.06271687296001291, 'l1_ratio': 0.04567705360840742}. Best is trial 11 with value: 0.2054330562824627.
[I 2026-10-08 19:49:34,468] Trial 27 finished with value: 0.2055879151587492 and parameters: {'alpha': 0.0009578298716025098, 'l1_ratio': 0.09320865777853499}. Best is 

In [102]:
results_df

,date,stock_id,R1M_Usd,y_pred,Lag1_R1M_Usd,Error
0,2012-11-30,24,0.048,0.014893,0.052,-0.033107
1,2012-11-30,34,0.114,0.014893,0.174,-0.099107
2,2012-11-30,41,-0.018,0.014893,0.001,0.032893
3,2012-11-30,45,-0.012,0.014893,-0.014,0.026893
4,2012-11-30,47,0.007,0.014893,-0.005,0.007893
...,...,...,...,...,...,...
11687,2018-12-31,1168,0.105,0.011827,-0.152,-0.093173
11688,2018-12-31,1186,0.080,0.024676,-0.172,-0.055324
11689,2018-12-31,1189,-0.103,0.008828,-0.394,0.111828
11690,2018-12-31,1195,0.157,0.005738,-0.180,-0.151262


In [103]:
hyperparameters_df

,date,alpha,l1_ratio,validation_mse
0,2012-11-30,1.719901,0.407379,0.011728
1,2013-05-31,0.007971,0.266929,0.010583
2,2013-11-30,0.003271,0.682060,0.010761
3,2014-05-31,0.002824,0.572148,0.007903
4,2014-11-30,0.003044,0.487749,0.007980
5,2015-05-31,0.004254,0.337541,0.007613
6,2015-11-30,0.003787,0.503768,0.011038
7,2016-05-31,0.002045,0.922799,0.012344
8,2016-11-30,0.002459,0.248734,0.013512
9,2017-05-31,0.001180,0.436726,0.013497


In [101]:
# Include random walk benchmark
# Random walk benchmark
results_df = results_df.merge(data_ml_selected_lags[['date', 'stock_id', 'Lag1_R1M_Usd']], on=['date', 'stock_id'], how='left')
results_df['Error'] = results_df['y_pred'] - results_df['R1M_Usd']
results_df

,date,stock_id,R1M_Usd,y_pred,Lag1_R1M_Usd,Error
0,2012-11-30,24,0.048,0.014893,0.052,-0.033107
1,2012-11-30,34,0.114,0.014893,0.174,-0.099107
2,2012-11-30,41,-0.018,0.014893,0.001,0.032893
3,2012-11-30,45,-0.012,0.014893,-0.014,0.026893
4,2012-11-30,47,0.007,0.014893,-0.005,0.007893
...,...,...,...,...,...,...
11687,2018-12-31,1168,0.105,0.011827,-0.152,-0.093173
11688,2018-12-31,1186,0.080,0.024676,-0.172,-0.055324
11689,2018-12-31,1189,-0.103,0.008828,-0.394,0.111828
11690,2018-12-31,1195,0.157,0.005738,-0.180,-0.151262


In [104]:
print(f'EN model:{mean_squared_error(results_df['y_pred'], results_df['R1M_Usd'])}')
print(f'RW model:{mean_squared_error(results_df['Lag1_R1M_Usd'], results_df['R1M_Usd'])}')

EN model:0.09098462189350821
RW model:0.1821770829627095
